In [ ]:
import numpy as np
from tensorflow import keras

INPUT_SHAPE = 10 #how many frames we use to make a predict
NUM_FEATURES = 21*8
NUM_CLASSES = 1001 #1000 + тишина

model = keras.Sequential(
    keras.layers.GRU(256, return_sequences=True, input_shape = (INPUT_SHAPE, NUM_FEATURES),
                     dropout = 0.25, recurrent_dropout = 0.2),
    keras.layers.GRU(128, return_sequences=False,
                    dropout=0.25, recurrent_dropout=0.2),
    keras.layers.GRU(64, return_sequences=False,
                     dropout=0.25, recurrent_dropout=0.2),

    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(NUM_CLASSES)
)

model.compile(optimizer='adam',
              loss='sparse_categorical',
              metrics=['accuracy'])



In [ ]:
import mediapipe as mp
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

class GuestureRecognizer:

    def __init__(self, sequence_length = 10, num_features = 21*8): #10 кадров = жест
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands = 1,
            min_detection_confidence = 0.5,
            min_tracking_confidence = 0.5
        )
        self.mp_drawnings = mp.solutions.drawning_utils
        self.sequence_length = sequence_length
        self.num_features = num_features  
        self.model = model
        self.is_trained = False
        self.landmarks_buffer = []
        self.current_sequence = []
        self.label_encoder = LabelEncoder()

    #код из handmarks but for only 1 video
    def extract_landmarks_from_video(self, video_path=0):
        landmarks_sequence = []
        cap = cv2.VideoCapture(video_path)
        
        prev_coords = np.zeros(4)  # x, y, z, visibility
        
        
        while True:
            success, frame = cap.read()
            if not success:
                break
                
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = self.hands.process(frame_rgb)

            
            frame_landmarks = []
            if results.multi_hand_landmarks:
                hand_landmarks = results.multi_hand_landmarks[0]
                
                for landmark in hand_landmarks.landmark:
                   
                    current_coords = np.array([
                        landmark.x, landmark.y, landmark.z, landmark.visibility
                    ])
                    
                    delta_coords = current_coords - prev_coords
                    
                    # Все признаки: текущие + изменения
                    frame_landmarks.extend(current_coords)
                    frame_landmarks.extend(delta_coords)
                    
                    prev_coords = current_coords
            else:
                frame_landmarks = [0.0] * self.num_features
                prev_coords = np.zeros(4)
            
            landmarks_sequence.append(frame_landmarks)
        
        cap.release()
        return landmarks_sequence

    def prepare_video_data_train(self, file_path = 'landmarks.npy'):
        
        data = np.load(file_path, allow_pickle=True).item()
        landmarks = data['landmarks']
        target = data['targets']
        video_train, video_test, target_train, target_test = train_test_split(
        landmarks, 
        target, 
        test_size=0.15, #по 3 видео для проверки на каждый класс
        random_state=42,
        stratify=target
        )

        target_train_encoded = self.label_encoder.fit_transform(target_train)
        target_test_encoded = self.label_encoder.transform(target_test)

        return video_train, video_test, target_train_encoded, target_test_encoded
    

    def prepare_for_online(self):
        """Подготовка для онлайн-режима с веб-камеры"""
        # Сброс буфера для новой сессии
        self.landmarks_buffer = []
        self.prev_coords = None
        print("✅ Онлайн-режим подготовлен. Буфер очищен.")

    def predict_online(self, draw=True):

        landmarks_sequence = self.extract_landmarks_from_video()
        if draw:
            self.mp_drawing.draw_landmarks(
                cv2.read(),
                landmarks_sequence, 
                self.mp_hands.HAND_CONNECTIONS 
            )
        self
        self.model.predict()
    








In [ ]:
import matplotlib.pyplot as plt

class Visualize:
    
    def plot_training_history(history):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        
        # Accuracy
        ax1.plot(history.history['accuracy'], label='Training Accuracy')
        ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
        ax1.set_title('Model Accuracy')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Accuracy')
        ax1.legend()
        
        # Loss
        ax2.plot(history.history['loss'], label='Training Loss')
        ax2.plot(history.history['val_loss'], label='Validation Loss')
        ax2.set_title('Model Loss')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.legend()
        
        plt.tight_layout()
        plt.show()


    def plot_cm_with_matplotlib(y_true, y_pred, class_names):
        cm = plt.confusion_matrix(y_true, y_pred)
        
        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
        ax.figure.colorbar(im, ax=ax)
        
        # Подписи осей
        ax.set(xticks=np.arange(cm.shape[1]),
               yticks=np.arange(cm.shape[0]),
               xticklabels=class_names,
               yticklabels=class_names,
               title='Confusion Matrix',
               ylabel='True Label',
               xlabel='Predicted Label')
        
        # Поворот подписей
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
        
        # Добавление чисел в ячейки
        thresh = cm.max() / 2.
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, format(cm[i, j], 'd'),
                       ha="center", va="center",
                       color="white" if cm[i, j] > thresh else "black")
        
        plt.tight_layout()
        plt.show()
        
        return cm

In [ ]:
from sklearn.model_selection import train_test_split

data = np.load('landmarks.npy', allow_pickle=True).item()
landmarks = data['landmarks']
target = data['targets']
video_train, video_test, target_train, target_test = train_test_split(
    landmarks, 
    target, 
    test_size=0.15, #по 3 видео для проверки на каждый класс
    random_state=42,
    stratify=target
)